In [2]:
from pathlib import Path
from PIL import Image, ImageOps

# =========================
# HEIC対応（pillow-heif）
# =========================
HEIF_AVAILABLE = False
try:
    import pillow_heif  # pip install pillow-heif
    pillow_heif.register_heif_opener()
    HEIF_AVAILABLE = True
except Exception:
    HEIF_AVAILABLE = False

# =========================
# 設定
# =========================
base_dir = Path(r"C:\Users\yamaz\Documents\GitHub\website\docs\img")
output_dir = base_dir / "crop-square"
output_dir.mkdir(exist_ok=True)

extensions = [".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp", ".heic"]

TARGET_WIDTH = 150
TARGET_HEIGHT = 150

print("=== Image Resize + Width Adjust Start ===")
print(f"📁 base_dir   : {base_dir}")
print(f"📁 output_dir : {output_dir}")
print(f"🧩 HEIC support: {'ON' if HEIF_AVAILABLE else 'OFF (install pillow-heif)'}\n")

# =========================
# メイン処理
# =========================
for img_path in base_dir.rglob("abstract*.png"):
    # フォルダは除外
    if img_path.is_dir():
        continue

    # 出力フォルダ配下は処理しない
    try:
        img_path.relative_to(output_dir)
        continue
    except ValueError:
        pass

    # 対象拡張子以外はスキップ
    if img_path.suffix.lower() not in extensions:
        continue

    # HEIC未対応ならスキップ
    if img_path.suffix.lower() == ".heic" and not HEIF_AVAILABLE:
        rel = img_path.relative_to(base_dir)
        print(f"❌ HEIC not supported (install pillow-heif): {rel}")
        continue

    rel = img_path.relative_to(base_dir)
    out_path = (output_dir / rel).with_suffix(".png")
    out_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        with Image.open(img_path) as img:
            # EXIF向き補正
            img = ImageOps.exif_transpose(img)

            # モード調整
            if img.mode == "P":
                img = img.convert("RGBA")
            elif img.mode not in ("RGB", "RGBA"):
                img = img.convert("RGBA")

            w, h = img.size

            # =========================
            # 1. 高さをTARGET_HEIGHTに揃える（アスペクト比維持）
            # =========================
            scale = TARGET_HEIGHT / h
            new_h = TARGET_HEIGHT
            new_w = int(w * scale)

            resized = img.resize((new_w, new_h), Image.LANCZOS)

            # =========================
            # 2. 幅をTARGET_WIDTHに調整
            #    - 大きい → 中央crop
            #    - 小さい → 白背景に中央配置
            # =========================
            if new_w > TARGET_WIDTH:
                left = (new_w - TARGET_WIDTH) // 2
                right = left + TARGET_WIDTH
                final_img = resized.crop((left, 0, right, TARGET_HEIGHT))

            elif new_w < TARGET_WIDTH:
                bg_mode = "RGBA" if resized.mode == "RGBA" else "RGB"
                bg_color = (255, 255, 255, 0) if bg_mode == "RGBA" else (255, 255, 255)
                canvas = Image.new(bg_mode, (TARGET_WIDTH, TARGET_HEIGHT), bg_color)

                paste_x = (TARGET_WIDTH - new_w) // 2

                if resized.mode == "RGBA":
                    canvas.paste(resized, (paste_x, 0), resized)
                else:
                    canvas.paste(resized, (paste_x, 0))

                final_img = canvas

            else:
                final_img = resized

            # 保存
            final_img.save(out_path, format="PNG", optimize=True)
            print(f"✅ Saved: {out_path}")

    except Exception as e:
        print(f"❌ Error processing {rel}: {e}")

print("\n=== Process Finished ===")

=== Image Resize + Width Adjust Start ===
📁 base_dir   : C:\Users\yamaz\Documents\GitHub\website\docs\img
📁 output_dir : C:\Users\yamaz\Documents\GitHub\website\docs\img\crop-square
🧩 HEIC support: ON

✅ Saved: C:\Users\yamaz\Documents\GitHub\website\docs\img\crop-square\abstract.png
✅ Saved: C:\Users\yamaz\Documents\GitHub\website\docs\img\crop-square\crop\abstract.png

=== Process Finished ===
